In [1]:
import pandas as pd
import numpy as np

previous_application = pd.read_csv("home-credit-default-risk/previous_application.csv")

In [2]:
missing_rate_percent = (previous_application.isnull().mean() * 100).sort_values(ascending=False)
print(missing_rate_percent)

RATE_INTEREST_PRIVILEGED       99.643698
RATE_INTEREST_PRIMARY          99.643698
AMT_DOWN_PAYMENT               53.636480
RATE_DOWN_PAYMENT              53.636480
NAME_TYPE_SUITE                49.119754
NFLAG_INSURED_ON_APPROVAL      40.298129
DAYS_TERMINATION               40.298129
DAYS_LAST_DUE                  40.298129
DAYS_LAST_DUE_1ST_VERSION      40.298129
DAYS_FIRST_DUE                 40.298129
DAYS_FIRST_DRAWING             40.298129
AMT_GOODS_PRICE                23.081773
AMT_ANNUITY                    22.286665
CNT_PAYMENT                    22.286366
PRODUCT_COMBINATION             0.020716
AMT_CREDIT                      0.000060
NAME_YIELD_GROUP                0.000000
NAME_PORTFOLIO                  0.000000
NAME_SELLER_INDUSTRY            0.000000
SELLERPLACE_AREA                0.000000
CHANNEL_TYPE                    0.000000
NAME_PRODUCT_TYPE               0.000000
SK_ID_PREV                      0.000000
NAME_GOODS_CATEGORY             0.000000
NAME_CLIENT_TYPE

In [3]:
status_counts = previous_application["NAME_CONTRACT_STATUS"].value_counts()
previous_application["NAME_CONTRACT_STATUS"] = previous_application["NAME_CONTRACT_STATUS"].apply(
    lambda x: x if status_counts[x] >= 5000 else "Other"
)
print(previous_application["NAME_CONTRACT_STATUS"].value_counts())

NAME_CONTRACT_STATUS
Approved        1036781
Canceled         316319
Refused          290678
Unused offer      26436
Name: count, dtype: int64


In [4]:
previous_application["credit_app_diff"] = previous_application["AMT_CREDIT"] - previous_application["AMT_APPLICATION"]
previous_application["credit_to_app_ratio"] = np.where(
    previous_application["AMT_APPLICATION"] > 0,
    previous_application["AMT_CREDIT"] / previous_application["AMT_APPLICATION"],
    np.nan,
)
previous_application["annuity_to_credit_ratio"] = np.where(
    previous_application["AMT_CREDIT"] > 0,
    previous_application["AMT_ANNUITY"] / previous_application["AMT_CREDIT"],
    np.nan,
)
previous_application["down_payment_ratio"] = np.where(
    previous_application["AMT_CREDIT"] > 0,
    previous_application["AMT_DOWN_PAYMENT"] / previous_application["AMT_CREDIT"],
    np.nan,
)
previous_application["is_last_application"] = (previous_application["FLAG_LAST_APPL_PER_CONTRACT"] == "Y").astype(int)

In [5]:
categorical_cols = ["NAME_CONTRACT_TYPE", "NAME_CONTRACT_STATUS"]

previous_encoded = pd.get_dummies(previous_application, columns=categorical_cols, drop_first=False, prefix_sep="__")

agg_dict = {
    "AMT_APPLICATION": ["mean", "max"],
    "AMT_CREDIT": ["mean", "max"],
    "AMT_ANNUITY": ["mean"],
    "AMT_GOODS_PRICE": ["mean"],
    "AMT_DOWN_PAYMENT": ["mean"],
    "credit_app_diff": ["mean"],
    "credit_to_app_ratio": ["mean"],
    "annuity_to_credit_ratio": ["mean"],
    "down_payment_ratio": ["mean"],
    "RATE_DOWN_PAYMENT": ["mean"],
    "CNT_PAYMENT": ["mean", "max"],
    "DAYS_DECISION": ["mean", "min", "max"],
    "DAYS_FIRST_DRAWING": ["min"],
    "DAYS_FIRST_DUE": ["min"],
    "DAYS_LAST_DUE": ["max"],
    "DAYS_LAST_DUE_1ST_VERSION": ["max"],
    "DAYS_TERMINATION": ["max"],
    "HOUR_APPR_PROCESS_START": ["mean"],
    "is_last_application": ["mean"],
    "NFLAG_LAST_APPL_IN_DAY": ["mean"],
    "NFLAG_INSURED_ON_APPROVAL": ["mean"],
}

for col in previous_encoded.columns:
    if col.startswith("NAME_CONTRACT_TYPE__") or col.startswith("NAME_CONTRACT_STATUS__"):
        agg_dict[col] = ["sum"]

prev_agg = previous_encoded.groupby("SK_ID_CURR").agg(agg_dict)
prev_agg.columns = [f"{col}_{stat}" for col, stat in prev_agg.columns]
prev_agg = prev_agg.reset_index()

prev_records = (
    previous_application.groupby("SK_ID_CURR").size().reset_index(name="prev_application_count")
)

prev_features = prev_agg.merge(prev_records, on="SK_ID_CURR", how="left")

status_cols = [
    col for col in prev_features.columns
    if col.startswith("NAME_CONTRACT_STATUS__") and col.endswith("_sum")
]

for status_col in status_cols:
    status_name = status_col.replace("NAME_CONTRACT_STATUS__", "").replace("_sum", "").lower()
    prev_features[f"prev_status_{status_name}"] = prev_features[status_col]

if status_cols:
    prev_features = prev_features.drop(columns=status_cols)

prev_features["prev_problem_applications"] = (
    prev_features.get("prev_status_refused", 0)
    + prev_features.get("prev_status_canceled", 0)
    + prev_features.get("prev_status_unused_offer", 0)
)

if "prev_status_approved" in prev_features.columns:
    prev_features["prev_approved_rate"] = (
        prev_features["prev_status_approved"] / prev_features["prev_application_count"]
    ).replace([np.inf, -np.inf], np.nan).fillna(0)
else:
    prev_features["prev_approved_rate"] = 0

if "prev_status_refused" in prev_features.columns:
    prev_features["prev_refused_rate"] = (
        prev_features["prev_status_refused"] / prev_features["prev_application_count"]
    ).replace([np.inf, -np.inf], np.nan).fillna(0)
else:
    prev_features["prev_refused_rate"] = 0

prev_features.head()


,SK_ID_CURR,AMT_APPLICATION_mean,AMT_APPLICATION_max,AMT_CREDIT_mean,AMT_CREDIT_max,AMT_ANNUITY_mean,AMT_GOODS_PRICE_mean,AMT_DOWN_PAYMENT_mean,credit_app_diff_mean,credit_to_app_ratio_mean,...,NAME_CONTRACT_TYPE__Revolving loans_sum,NAME_CONTRACT_TYPE__XNA_sum,prev_application_count,prev_status_approved,prev_status_canceled,prev_status_refused,prev_status_unused offer,prev_problem_applications,prev_approved_rate,prev_refused_rate
0,100001,24835.50,24835.5,23787.00,23787.0,3951.000,24835.5,2520.0,-1048.5,0.957782,...,0,0,1,1,0,0,0,0,1.0,0.0
1,100002,179055.00,179055.0,179055.00,179055.0,9251.775,179055.0,0.0,0.0,1.000000,...,0,0,1,1,0,0,0,0,1.0,0.0
2,100003,435436.50,900000.0,484191.00,1035882.0,56553.990,435436.5,3442.5,48754.5,1.057664,...,0,0,3,3,0,0,0,0,1.0,0.0
3,100004,24282.00,24282.0,20106.00,20106.0,5357.250,24282.0,4860.0,-4176.0,0.828021,...,0,0,1,1,0,0,0,0,1.0,0.0
4,100005,22308.75,44617.5,20076.75,40153.5,4813.200,44617.5,4464.0,-2232.0,0.899950,...,0,0,2,1,1,0,0,1,0.5,0.0


In [6]:
pd.DataFrame.to_csv(prev_features, "transformed_data/_prev_features.csv") 